# Initial Setup

In [1]:
%pip install catboost lightgbm optuna holidays duckdb

## Imports

In [2]:
import numpy as np
import pandas as pd
import duckdb

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

## Variables

In [3]:
def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

DRIVE_FOLDER_NAME = "Store Sales"

In [4]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [5]:
TARGET = "sales"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']


## Helper Functions

### Data Loader

In [6]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/train.csv')
    test_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

### Loader of additional data

In [7]:
def load_helper_data(option: str="local"):
  '''
  returns oil_df, transactions_df, holidays_events_df, stores_df

  '''
  if option == "local":
    oil_df = pd.read_csv('oil.csv')
    transactions_df = pd.read_csv('transactions.csv')
    holidays_events_df = pd.read_csv('holidays_events.csv')
    stores_df = pd.read_csv('stores.csv')


  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    oil_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/oil.csv')
    transactions_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/transactions.csv')
    holidays_events_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/holidays_events.csv')
    stores_df = pd.read_csv(f'/content/drive/MyDrive/Kaggle Practice/{DRIVE_FOLDER_NAME}/stores.csv')
    
  oil_df['date'] =  pd.to_datetime(oil_df['date'])
  transactions_df['date'] =  pd.to_datetime(transactions_df['date'])
  holidays_events_df['date'] =  pd.to_datetime(holidays_events_df['date'])
  


  return oil_df, transactions_df, holidays_events_df, stores_df

### Data Splitter

In [8]:
def data_splitter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

## Create DataFrames

In [9]:
train_df, test_df, df = load_data(ENVIRONMENT)
oil_df, transactions_df, holidays_events_df, stores_df = load_helper_data(ENVIRONMENT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# EDA

## Overall Info

### Main DF

In [0]:
df.head()

In [0]:
df.groupby('source').agg(min_date=('date', 'min'), max_date=('date', 'max'))

In [0]:
df.describe(include='all')

### Stores

In [0]:
stores_df.head()

In [0]:
stores_df.describe(include='all')

### Transactions

In [0]:
transactions_df.head()

In [0]:
transactions_df.describe(include='all')

### Holidays

In [0]:
holidays_events_df.head()

In [0]:
holidays_events_df.describe()

### Oil

In [0]:
oil_df.head()

In [0]:
oil_df.describe()

In [0]:
df.head()

In [0]:
transactions_df

# Merging datasets

In [23]:
duckdb.sql("""--sql
    SELECT
        SUM(CASE WHEN sales = 0 THEN 1 ELSE 0 END) AS zero_sales,
        SUM(CASE WHEN sales > 0 THEN 1 ELSE 0 END) AS more_sales
    FROM temp_df
    WHERE transactions IS NULL
""").df()

,zero_sales,more_sales
0,242536.0,3248.0


In [0]:
mask = temp_df["transactions"].notna()

ratio_table = temp_df.loc[mask].groupby(["store_nbr", "family"]).agg(
    total_sales=("sales", "sum"),
    total_transactions=("transactions", 'sum')
).reset_index()

ratio_table['ratio'] = ratio_table['total_sales'] / ratio_table['total_transactions']
display(ratio_table)

In [17]:
is_transferred = holidays_events_df['transferred'] == True
legit_holidays = holidays_events_df['type'].isin(['Holiday', 'Transfer', 'Bridge', 'Additional'])
is_workday = holidays_events_df['type']  == 'Work Day'
is_event = holidays_events_df['type']  == 'Event'

is_national = holidays_events_df['locale'] == 'National'
is_regional = holidays_events_df['locale'] == 'Regional'
is_local = holidays_events_df['locale'] == 'Local'

national_holidays = holidays_events_df.loc[(~is_transferred) & (is_national) & (legit_holidays)]
regional_holidays = holidays_events_df.loc[(~is_transferred) & (is_regional) & (legit_holidays)]
local_holidays = holidays_events_df.loc[(~is_transferred) & (is_local) & (legit_holidays)]

national_holidays = national_holidays.groupby('date').agg(nat_num_holidays=('type', 'count'), nat_holiday_types =('type', ', '.join)).sort_values(by='nat_num_holidays', ascending=False).reset_index()
national_holidays['is_national'] = 1

regional_holidays = regional_holidays.groupby(['date', 'locale_name']).agg(reg_num_holidays=('type', 'count'), reg_holiday_types =('type', ', '.join)).sort_values(by='reg_num_holidays', ascending=False).reset_index()
regional_holidays['is_regional'] = 1

local_holidays = local_holidays.groupby(['date', 'locale_name']).agg(loc_num_holidays=('type', 'count'), local_holiday_types =('type', ', '.join)).sort_values(by='loc_num_holidays', ascending=False).reset_index()
local_holidays['is_local'] = 1


workdays = holidays_events_df.loc[(~is_transferred) & (is_workday)].copy()
workdays['is_workday'] = 1
workdays = workdays[['date', 'is_workday']].drop_duplicates(subset=['date'], keep='first')

events = holidays_events_df.loc[(~is_transferred) & (is_event)].copy()
events['is_event'] = 1
events = events[['date', 'is_event']].drop_duplicates(subset=['date'], keep='first')

In [18]:
local_holidays

,date,locale_name,loc_num_holidays,local_holiday_types,is_local
0,2016-07-24,Guayaquil,2,"Additional, Transfer",1
1,2012-04-12,Cuenca,1,Holiday,1
2,2012-03-02,Manta,1,Holiday,1
3,2012-04-21,Riobamba,1,Holiday,1
4,2012-05-12,Puyo,1,Holiday,1
...,...,...,...,...,...
142,2017-11-12,Ambato,1,Holiday,1
143,2017-12-05,Quito,1,Additional,1
144,2017-12-08,Loja,1,Holiday,1
145,2017-12-08,Quito,1,Transfer,1


In [ ]:
initial_length = len(df)

oil_df = (oil_df.set_index("date")
                .reindex(pd.date_range(oil_df["date"].min(), df["date"].max()))
                .ffill().bfill()
                .rename_axis("date").reset_index())

temp_df = (df.merge(transactions_df, on=['date', 'store_nbr'], how='left')
            .merge(stores_df, on='store_nbr', how='left')
            .merge(oil_df, on='date', how='left')
            .merge(national_holidays, on='date', how='left')
            .merge(regional_holidays, left_on=['date', 'state'], right_on=['date','locale_name'], how='left').drop(columns=['locale_name'])
            .merge(local_holidays, left_on=['date','city'], right_on=['date','locale_name'], how='left').drop(columns=['locale_name'])
            .merge(workdays, on='date', how='left')
						.merge(events, on='date', how='left')
            )

In [21]:
assert initial_length == len(temp_df), 'Not equal'

In [22]:
temp_df

,id,date,store_nbr,family,sales,onpromotion,source,transactions,city,state,...,nat_holiday_types,is_national,reg_num_holidays,reg_holiday_types,is_regional,loc_num_holidays,local_holiday_types,is_local,is_workday,is_event
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,train,NaN,Quito,Pichincha,...,Holiday,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,train,NaN,Quito,Pichincha,...,Holiday,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,train,NaN,Quito,Pichincha,...,Holiday,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,train,NaN,Quito,Pichincha,...,Holiday,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,train,NaN,Quito,Pichincha,...,Holiday,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3029395,3029395,2017-08-31,9,POULTRY,NaN,1,test,NaN,Quito,Pichincha,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3029396,3029396,2017-08-31,9,PREPARED FOODS,NaN,0,test,NaN,Quito,Pichincha,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3029397,3029397,2017-08-31,9,PRODUCE,NaN,1,test,NaN,Quito,Pichincha,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3029398,3029398,2017-08-31,9,SCHOOL AND OFFICE SUPPLIES,NaN,9,test,NaN,Quito,Pichincha,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [0]:
stores_df['city'].unique()

In [0]:
display(stores_df)

In [0]:
df.sort_values(by='date', ascending=True).reset_index(drop=True).head(10)

In [0]:
df.index[df['date'] == '2017-01-01']

In [0]:
dates = np.sort(df["date"].unique())

tscv = TimeSeriesSplit(n_splits=4, test_size=16)   # 16 days = competition horizon

for i in enumerate(tscv.split(dates)):
    print(i)

In [0]:
stores_df['state'].unique()